In [9]:
pip install --upgrade unsloth unsloth_zoo trl transformers accelerate

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.7/53.7 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.7/62.7 MB 31.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 401.6/401.6 kB 23.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 32.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 100.6 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 21.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 32.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 27.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 85.4 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 91.3 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 83.6 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 224.9/224.9 kB 18.2 MB/s eta 0:00:

In [10]:
from unsloth import FastLanguageModel
from datasets import load_dataset, Dataset
from trl import SFTTrainer, SFTConfig
from transformers import TextStreamer

max_seq_length = 2048
lora_rank = 16

# ---------------------------------------------------------------------------
# Model + LoRA
# ---------------------------------------------------------------------------
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen3-4B",
    max_seq_length = max_seq_length,
    load_in_4bit = True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = lora_rank,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = lora_rank * 2,
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

# ---------------------------------------------------------------------------
# Dataset
# ---------------------------------------------------------------------------
SYSTEM_PROMPT = "You are an expert medical doctor. Answer the patient's question with a clear diagnosis and treatment advice."

raw = load_dataset(
    "Malikeh1375/medical-question-answering-datasets",
    name = "chatdoctor_healthcaremagic",
    split = "train",
)


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.3.8: Fast Qwen3 patching. Transformers: 5.3.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/3.55G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/237 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

unsloth/qwen3-4b-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Unsloth 2026.3.8 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


README.md: 0.00B [00:00, ?B/s]

chatdoctor_healthcaremagic/train-00000-o(…):   0%|          | 0.00/70.5M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/112165 [00:00<?, ? examples/s]

In [11]:
raw = raw.select(range(5000))

In [12]:
print(raw[0])

{'instruction': "If you are a doctor, please answer the medical questions based on the patient's description.", 'input': 'I woke up this morning feeling the whole room is spinning when i was sitting down. I went to the bathroom walking unsteadily, as i tried to focus i feel nauseous. I try to vomit but it wont come out.. After taking panadol and sleep for few hours, i still feel the same.. By the way, if i lay down or sit down, my head do not spin, only when i want to move around then i feel the whole world is spinning.. And it is normal stomach discomfort at the same time? Earlier after i relieved myself, the spinning lessen so i am not sure whether its connected or coincidences.. Thank you doc!', 'output': 'Hi, Thank you for posting your query. The most likely cause for your symptoms is benign paroxysmal positional vertigo (BPPV), a type of peripheral vertigo. In this condition, the most common symptom is dizziness or giddiness, which is made worse with movements. Accompanying nausea

In [13]:
def format_row(x):
    question = x["input"].strip()
    return tokenizer.apply_chat_template([
        {"role": "system",    "content": SYSTEM_PROMPT},
        {"role": "user",      "content": question},
        {"role": "assistant", "content": x["output"].strip()},
    ], tokenize=False)

raw = raw.map(lambda x: {"text": format_row(x)},
              remove_columns=raw.column_names)

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

In [17]:
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = raw,
    args = SFTConfig(
        dataset_text_field = "text",
        max_seq_length = max_seq_length,
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 10,
        num_train_epochs = 2,
        eval_steps = 100,
        learning_rate = 1e-4,
        logging_steps = 25,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "cosine",
        seed = 3407,
        output_dir = "outputs",
        # report_to = "",
    ),
)
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 5,000 | Num Epochs = 2 | Total steps = 1,250
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 33,030,144 of 4,055,498,240 (0.81% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
25,3.185106
50,2.465156
75,2.238671
100,2.316499
125,2.278904
150,2.239111
175,2.269049
200,2.265233
225,2.183831
250,2.209932


TrainOutput(global_step=1250, training_loss=2.172137255859375, metrics={'train_runtime': 5260.4827, 'train_samples_per_second': 1.901, 'train_steps_per_second': 0.238, 'total_flos': 5.894774499474432e+16, 'train_loss': 2.172137255859375, 'epoch': 2.0})

In [19]:
# ---------------------------------------------------------------------------
model.save_pretrained("medical_lora")
tokenizer.save_pretrained("medical_lora")


('medical_lora/tokenizer_config.json',
 'medical_lora/chat_template.jinja',
 'medical_lora/tokenizer.json')

In [26]:
from unsloth import FastLanguageModel
from transformers import TextStreamer

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "medical_lora",   # path to saved LoRA
    # max_seq_length = 2048,
    # load_in_4bit = True,
)


==((====))==  Unsloth 2026.3.8: Fast Qwen3 patching. Transformers: 5.3.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

unsloth/qwen3-4b-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


In [49]:
import torch
from unsloth import FastLanguageModel

# Load the merged model through Unsloth — avoids all attribute-missing issues
# because Unsloth initialises its own fast-inference state during from_pretrained.
model, tokenizer = FastLanguageModel.from_pretrained(
    "medical_lora",       # LoRA checkpoint — Unsloth reads adapter_config.json, loads base + adapters
    max_seq_length = 2048,
    load_in_4bit   = True,            # Unsloth uses double-quant internally → fast_gemv works
    dtype          = torch.float16,   # T4 doesn't support bf16
)
model.eval()   # training mode off; skip for_inference to avoid Unsloth's buggy fast path

SYSTEM_PROMPT = "You are an expert medical doctor. Answer the patient's question with a clear diagnosis and treatment advice."

def ask(question: str):
    from transformers import TextStreamer
    text = tokenizer.apply_chat_template([
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": question},
    ], tokenize=False, add_generation_prompt=True)

    inputs = tokenizer(text, return_tensors="pt").to("cuda:0")
    with torch.no_grad():
        _ = model.generate(
            **inputs,
            max_new_tokens = 512,
            temperature    = 0.7,
            do_sample      = True,
            use_cache      = False,   # keeps past_key_values=None → skips Unsloth's buggy fast_forward_inference
            streamer       = TextStreamer(tokenizer, skip_prompt=True),
        )

# --- test queries ---
ask("I have had a fever of 39°C, sore throat, and fatigue for 3 days. What should I do?")
ask("I am a 45-year-old male with high blood pressure. Can I take ibuprofen?")


==((====))==  Unsloth 2026.3.8: Fast Qwen3 patching. Transformers: 5.3.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

unsloth/qwen3-4b-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Both `max_new_tokens` (=512) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)


<think>

</think>

Hi, Welcome to Chat Doctor, Fever with sore throat can be due to viral or bacterial infection. If you have a sore throat, it is very important to know if it is a viral or bacterial sore throat, because you need different treatments for the two. Viral sore throat can be treated at home with rest, cold sponging, analgesics like acetaminophen or ibuprofen, and plenty of water. If the sore throat is bacterial then you need to take antibiotics. Hope this may help you. Let me know if you have any other query.<|im_end|>


Both `max_new_tokens` (=512) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


<think>

</think>

Hello, Thank you for asking at Chat Doctor. I went through your history and would like to make suggestions for you as follows<|im_end|>


In [51]:
ask("my nose is blocked due to weather change, i take oritivin to fix nose but i do not have it, what should i take")


Both `max_new_tokens` (=512) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


<think>

</think>

Hi, Thanks for your query. I have gone through your query. Oritavine is a nasal spray containing a combination of antiallergic Chat Doctor.  I suggest you to use any other antiallergic nasal spray like Neti Chat Doctor.  I suggest you to avoid exposure to dust and pollen. If symptoms persist you can use oral antihistamine like Levocetrizine. I hope my answer will help you. Take care.<|im_end|>


In [52]:
import shutil
shutil.make_archive("/kaggle/working/medical_lora_zip", "zip", "/kaggle/working/medical_lora")


'/kaggle/working/medical_lora_zip.zip'

In [ ]:
import shutil
shutil.make_archive("/kaggle/working/medical_lora_zip", "zip", "/kaggle/working/medical_lora")


In [ ]:
from huggingface_hub import HfApi
api = HfApi()
api.upload_folder(
    folder_path="medical_lora",
    repo_id="KingLLM/medical-finetuned",
    repo_type="model",
    token="hf-",
)


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

CommitInfo(commit_url='https://huggingface.co/KingLLM/medical-finetuned/commit/07d783a615ad0a8227c1afc241fb0a91c3c759e3', commit_message='Upload folder using huggingface_hub', commit_description='', oid='07d783a615ad0a8227c1afc241fb0a91c3c759e3', pr_url=None, repo_url=RepoUrl('https://huggingface.co/KingLLM/medical-finetuned', endpoint='https://huggingface.co', repo_type='model', repo_id='KingLLM/medical-finetuned'), pr_revision=None, pr_num=None)